# XGBoost and neural ranking factorial

This notebook defines and reports the new **68-model** experiment without modifying the earlier frozen-BGE, metadata-MLP, or XGBoost notebooks.

- XGBoost: 2 feature sets x 2 audience-draw policies = 4 variants.
- Neural pairwise rankers: 2 x 2 x 2 negative samplers x 2 schedules x 2 heads x 2 network sizes = 64 variants.
- Every variant runs five-fold development CV, a final development refit, and sealed Paper 2 test inference for the selected scopes.
- Every variant has a stable ID and independent signed caches/checkpoints.

For XGBoost, `mean10` uses integer selection counts (0-10) with linear-gain `rank:ndcg`. This is a memory-safe model of average selection frequency; it is not mathematically identical to averaging ten separate binary LambdaRank objectives. Fractional labels are invalid in XGBoost LambdaRank. Neural `mean10` averages pairwise training evidence over the ten binary draws.

Each XGBoost scope independently uses the original two-stage tuning design: 32 broad random configurations x 5 development folds, followed by a 27-configuration narrow refinement grid x 5 folds. The configuration with the highest mean development-CV macro nDCG@k is refitted on all development articles and then evaluated once on the sealed test set.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import pandas as pd

from commentgap_analysis.factorial_rankers import factorial_variants

OUTPUT_ROOT = Path("model_output/selection_2025/factorial_rankers")
variants = pd.DataFrame([asdict(variant) for variant in factorial_variants()])
print(f"{len(variants)} variants")
display(variants)

## Run from a terminal

```bash
.venv/bin/python -u scripts/run_ranker_factorial.py 2>&1 | tee logs/06C5_ranker_factorial.log
```

Preview the plan without training:

```bash
.venv/bin/python -u scripts/run_ranker_factorial.py --plan
```

Resume while skipping slow classes or individual models (filters are repeatable shell globs):

```bash
.venv/bin/python -u scripts/run_ranker_factorial.py \
  --exclude 'xgb__metadata_bge__*' \
  --exclude 'nn__metadata_bge__mean10__*'
```

Run only selected variants or one scope:

```bash
.venv/bin/python -u scripts/run_ranker_factorial.py \
  --include 'nn__metadata__draw1__*__base' \
  --scope root
```

Completed work is reused automatically. Neural work resumes at epoch boundaries; XGBoost CV resumes at completed-fold boundaries; the shared XGBoost BGE cache resumes while it is being built. Use `--list-models` for every stable ID and `--force-recompute` only for an intentional clean rerun.

Redo only the four independently tuned XGBoost variants while leaving all completed neural outputs untouched:

```bash
.venv/bin/python -u scripts/run_ranker_factorial.py \
  --include 'xgb__*' \
  --xgb-device cpu
```

## Recorded results

The runner refreshes these experiment-wide tables after every completed or failed model:

- `experiment_variants.csv`: all axes, scopes, status, and output paths.
- `development_cv_results.csv`: fold-level CV nDCG and training-selection results.
- `development_training_results.csv`: final-refit epoch histories or XGBoost fit summaries.
- `sealed_test_results.csv`: selector- and tie-draw-specific sealed-test metrics.

Model-specific checkpoints, histories, predictions, metrics, and manifests remain under `<output root>/<variant ID>/<scope>/`. Test labels never drive CV, early stopping, or model selection.

In [ ]:
def read_if_present(name):
    path = OUTPUT_ROOT / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

experiment_status = read_if_present("experiment_variants.csv")
cv_results = read_if_present("development_cv_results.csv")
training_results = read_if_present("development_training_results.csv")
test_results = read_if_present("sealed_test_results.csv")

print({
    "status_rows": len(experiment_status),
    "cv_rows": len(cv_results),
    "training_rows": len(training_results),
    "test_rows": len(test_results),
})
display(experiment_status)

In [ ]:
if not cv_results.empty:
    cv_summary = (
        cv_results.groupby(["variant_id", "family", "scope"], as_index=False)
        .agg(
            cv_mean_ndcg_at_k=("macro_ndcg_at_k", "mean"),
            cv_sd_ndcg_at_k=("macro_ndcg_at_k", "std"),
            completed_folds=("fold", "nunique"),
        )
        .sort_values(["scope", "cv_mean_ndcg_at_k"], ascending=[True, False])
    )
    display(cv_summary)

In [ ]:
if not test_results.empty:
    test_summary = (
        test_results.groupby(
            ["variant_id", "family", "scope", "selector"],
            as_index=False,
        )
        .agg(
            ten_draw_mean_ndcg_at_k=("ndcg_at_k", "mean"),
            draw_sd_ndcg_at_k=("ndcg_at_k", "std"),
        )
        .sort_values(
            ["scope", "selector", "ten_draw_mean_ndcg_at_k"],
            ascending=[True, True, False],
        )
    )
    display(test_summary)